In [12]:
sample_review = "The Samsung phone has a poor camera and battery issue."

result = analyze_review(sample_review)

result

{'review': 'The Samsung phone has a poor camera and battery issue.',
 'clean_review': 'the samsung phone has a poor camera and battery issue',
 'entities': [{'text': 'Samsung', 'label': 'ORG'}],
 'domain_matches': [{'text': 'Samsung', 'category': 'BRANDS'}],
 'keywords': [{'keyword': 'camera', 'score': 0.4983},
  {'keyword': 'samsung', 'score': 0.4496},
  {'keyword': 'poor', 'score': 0.4496},
  {'keyword': 'phone', 'score': 0.4496},
  {'keyword': 'battery', 'score': 0.381}]}

In [11]:
def analyze_review(review):
    clean_review = preprocess_text(review)

    doc = nlp(review)

    entities = [
        {
            "text": ent.text,
            "label": ent.label_
        }
        for ent in doc.ents
    ]

    matches = matcher(doc)

    domain_matches = [
        {
            "text": doc[start:end].text,
            "category": nlp.vocab.strings[match_id]
        }
        for match_id, start, end in matches
    ]

    review_vector = vectorizer.transform([clean_review])
    scores = review_vector.toarray().flatten()

    top_indices = scores.argsort()[-10:][::-1]

    keywords = [
        {
            "keyword": feature_names[index],
            "score": round(float(scores[index]), 4)
        }
        for index in top_indices
        if scores[index] > 0
    ]

    return {
        "review": review,
        "clean_review": clean_review,
        "entities": entities,
        "domain_matches": domain_matches,
        "keywords": keywords
    }

In [10]:
vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(df["clean_review"])

feature_names = vectorizer.get_feature_names_out()

review_index = 0
scores = tfidf_matrix[review_index].toarray().flatten()

top_indices = scores.argsort()[-10:][::-1]

keywords = [
    {
        "keyword": feature_names[index],
        "score": round(float(scores[index]), 4)
    }
    for index in top_indices
    if scores[index] > 0
]

keywords

[{'keyword': 'drains quickly', 'score': 0.3432},
 {'keyword': 's24 battery', 'score': 0.3432},
 {'keyword': 'quickly', 'score': 0.3017},
 {'keyword': 'samsung galaxy', 'score': 0.3017},
 {'keyword': 'galaxy s24', 'score': 0.3017},
 {'keyword': 'drains', 'score': 0.3017},
 {'keyword': 's24', 'score': 0.3017},
 {'keyword': 'galaxy', 'score': 0.3017},
 {'keyword': 'battery drains', 'score': 0.3017},
 {'keyword': 'samsung', 'score': 0.2722}]

In [9]:
DOMAIN_PATH = BASE_DIR / "data" / "domain_terms.json"

with open(DOMAIN_PATH, "r", encoding="utf-8") as file:
    domain_terms = json.load(file)

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

for category, terms in domain_terms.items():
    patterns = [nlp.make_doc(term) for term in terms]
    matcher.add(category.upper(), patterns)

doc = nlp(sample_review)
matches = matcher(doc)

matched_phrases = [
    {
        "text": doc[start:end].text,
        "category": nlp.vocab.strings[match_id]
    }
    for match_id, start, end in matches
]

matched_phrases

[{'text': 'Samsung', 'category': 'BRANDS'},
 {'text': 'Galaxy S24', 'category': 'PRODUCTS'}]

In [8]:
nlp = spacy.load("en_core_web_sm")

sample_review = df["review"].iloc[0]
doc = nlp(sample_review)

entities = [
    {
        "text": ent.text,
        "label": ent.label_
    }
    for ent in doc.ents
]

print("Review:", sample_review)
print("Entities:", entities)

Review: Samsung Galaxy S24 battery drains quickly.
Entities: [{'text': 'Samsung Galaxy S24', 'label': 'ORG'}]


In [7]:
df["clean_review"] = df["review"].apply(preprocess_text)

df[["review", "clean_review"]].head()

,review,clean_review
0,Samsung Galaxy S24 battery drains quickly.,samsung galaxy s24 battery drains quickly
1,The Samsung camera quality is excellent.,the samsung camera quality is excellent
2,My iPhone 15 works perfectly and the battery l...,my iphone 15 works perfectly and the battery l...
3,OnePlus phone overheats after the latest update.,oneplus phone overheats after the latest update
4,AirPods Pro sound quality is amazing.,airpods pro sound quality is amazing


In [6]:
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df["sentiment"].value_counts()

Dataset shape: (20, 3)
Columns: ['review_id', 'review', 'sentiment']


sentiment
negative    10
positive    10
Name: count, dtype: int64

In [5]:
BASE_DIR = Path.cwd()

if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "data" / "product_reviews.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,review_id,review,sentiment
0,1,Samsung Galaxy S24 battery drains quickly.,negative
1,2,The Samsung camera quality is excellent.,positive
2,3,My iPhone 15 works perfectly and the battery l...,positive
3,4,OnePlus phone overheats after the latest update.,negative
4,5,AirPods Pro sound quality is amazing.,positive


In [4]:
print(os.getcwd())

d:\natural-language-processing\day-5\notebooks


In [3]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
import spacy

from spacy.matcher import PhraseMatcher
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.append(os.path.abspath(".."))

from src.preprocess import preprocess_text